# 01 - Ollama Tool Loop Skeleton (Verbose Trace)

This notebook is the smallest end-to-end example of the tool loop pattern used in this repo.

Reading order:
1. Configure imports + adapter + trace settings.
2. Define toy world state + tool functions + JSON schemas.
3. Define shared trace helpers and robust tool-call parsing.
4. Run the iterative loop (`model -> tool calls -> tool results -> repeat`).
5. Execute a demo run and inspect full trace snapshots.

What to watch:
- `ASSISTANT-THINKING` lines show visible reasoning text from the model response.
- `TOOL-SOURCE`, `TOOL-CALL`, and `TOOL-RESULT` show exactly what was requested and executed.
- `RAW-RESPONSE` shows the full payload returned by the adapter/Ollama client.


## Step 1 - Environment and Adapter Setup

This cell bootstraps repo imports, sets model/runtime constants, and instantiates `LLMAdapter`.

Key idea: keep all model/trace toggles centralized so behavior is easy to tune.


In [11]:
# Section: Setup
# Purpose: Import dependencies, configure tracing, and initialize the LLM adapter.

import json
import random
import re
import sys
import time
from pathlib import Path
from typing import Any

# Ensure repo root is importable when running this notebook from its folder.
repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "orchestrator").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from orchestrator.llm_interaction.adapter import LLMAdapter, LLMError

MODEL = "gpt-oss:20b"
MAX_ITERATIONS = 60
TRACE_PREVIEW_CHARS = 500

# Maximum-detail tracing controls
FULL_TRACE = True
SHOW_THINKING_TRACE = True
SHOW_RAW_RESPONSE = True
SHOW_RESPONSE_STATS = True
PRINT_FULL_TRACE_AFTER_RUN = True
PRINT_FULL_RESPONSE_SNAPSHOTS_AFTER_RUN = True

adapter = LLMAdapter(model=MODEL, verbose=False)


## Step 2 - World State and Tool Registry

This cell defines a minimal DM state and exposes tools the model can call.

Key idea: tools are regular Python functions plus JSON schemas in `TOOLS`.


In [7]:
# Section: Domain state + tools
# Purpose: Define world data, tool implementations, and tool schemas exposed to the model.

world_state = {
    "location": "Ravenhill",
    "time_of_day": "dusk",
    "weather": "foggy",
    "active_quests": ["Find the missing miller"],
}


def get_scene_state() -> dict[str, Any]:
    return world_state


def set_scene_state(
    location: str | None = None,
    time_of_day: str | None = None,
    weather: str | None = None,
) -> dict[str, Any]:
    if location:
        world_state["location"] = location
    if time_of_day:
        world_state["time_of_day"] = time_of_day
    if weather:
        world_state["weather"] = weather
    return world_state


def roll_dice(expression: str = "1d20") -> dict[str, Any]:
    m = re.fullmatch(r"(\d+)d(\d+)([+-]\d+)?", expression.strip())
    if not m:
        raise ValueError(f"Invalid dice expression: {expression}")
    n, sides, mod = int(m.group(1)), int(m.group(2)), int(m.group(3) or 0)
    rolls = [random.randint(1, sides) for _ in range(n)]
    return {
        "expression": expression,
        "rolls": rolls,
        "modifier": mod,
        "total": sum(rolls) + mod,
    }


def add_quest(title: str) -> dict[str, Any]:
    world_state.setdefault("active_quests", []).append(title)
    return world_state


TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_scene_state",
            "description": "Read the current DM world state.",
            "parameters": {"type": "object", "properties": {}, "required": []},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "set_scene_state",
            "description": "Update location/time/weather.",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {"type": "string"},
                    "time_of_day": {"type": "string"},
                    "weather": {"type": "string"},
                },
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "roll_dice",
            "description": "Roll dice in NdM(+/-K) format, e.g. 1d20+3.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "Dice expression"}
                },
                "required": ["expression"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "add_quest",
            "description": "Add a new quest objective to the active quest log.",
            "parameters": {
                "type": "object",
                "properties": {
                    "title": {"type": "string"}
                },
                "required": ["title"],
            },
        },
    },
]

TOOL_IMPL = {
    "get_scene_state": get_scene_state,
    "set_scene_state": set_scene_state,
    "roll_dice": roll_dice,
    "add_quest": add_quest,
}


## Step 3 - Trace and Tool Execution Helpers

These helpers standardize log formatting and make tool extraction resilient.

Key idea: we normalize structured tool calls first, then attempt text-fallback parsing.


In [8]:
# Section: Runtime helpers
# Purpose: Provide structured tracing, response normalization, and safe tool execution wrappers.

RUN_STATE = {
    "total_tool_calls": 0,
}


def reset_run_state() -> None:
    RUN_STATE["total_tool_calls"] = 0


def ts() -> str:
    return time.strftime("%H:%M:%S")


def preview(text: str, limit: int = TRACE_PREVIEW_CHARS) -> str:
    if text is None:
        return ""
    text = str(text).strip()
    if FULL_TRACE:
        return text
    if len(text) <= limit:
        return text
    return text[:limit] + f" ... [truncated {len(text) - limit} chars]"


def trace_print(tag: str, message: str, trace_lines: list[str] | None = None) -> None:
    line = f"[{ts()}] [{tag}] {message}"
    print(line, flush=True)
    if trace_lines is not None:
        trace_lines.append(line)


def response_to_dict(response: Any) -> dict[str, Any]:
    if hasattr(response, "model_dump"):
        try:
            data = response.model_dump(exclude_none=False)
            if isinstance(data, dict):
                return data
        except Exception:
            pass

    if isinstance(response, dict):
        return response

    return {
        "_type": type(response).__name__,
        "repr": repr(response),
    }


def extract_message_dict(response: Any) -> dict[str, Any]:
    data = response_to_dict(response)
    msg = data.get("message") if isinstance(data, dict) else None
    if isinstance(msg, dict):
        return msg
    if hasattr(msg, "model_dump"):
        try:
            dumped = msg.model_dump(exclude_none=False)
            if isinstance(dumped, dict):
                return dumped
        except Exception:
            pass
    return {}


def extract_thinking_text(response: Any) -> str:
    msg = extract_message_dict(response)
    thinking = msg.get("thinking", "")

    if isinstance(thinking, list):
        return "".join(str(x) for x in thinking)

    return str(thinking or "")


def normalize_tool_calls(response: Any) -> list[dict[str, Any]]:
    out: list[dict[str, Any]] = []
    raw_calls = adapter._extract_tool_calls(response)

    for i, call in enumerate(raw_calls):
        if not isinstance(call, dict):
            continue

        fn = call.get("function", {})
        name = fn.get("name") or call.get("name")
        args = fn.get("arguments", {})

        if isinstance(args, str):
            try:
                args = json.loads(args)
            except json.JSONDecodeError:
                args = {}

        if not isinstance(args, dict):
            args = {}

        if not name:
            continue

        out.append(
            {
                "id": call.get("id") or f"call_{i}",
                "name": str(name),
                "arguments": args,
                "raw": call,
                "source": "structured",
            }
        )

    return out


def extract_fallback_tool_calls_from_text(text: str) -> list[dict[str, Any]]:
    if not text:
        return []

    calls: list[dict[str, Any]] = []

    for line in text.splitlines():
        stripped = line.strip().rstrip(",")
        if not stripped.startswith("{") or not stripped.endswith("}"):
            continue
        try:
            obj = json.loads(stripped)
        except json.JSONDecodeError:
            continue
        if not isinstance(obj, dict):
            continue
        name = obj.get("name")
        params = obj.get("parameters", {})
        if not name or not isinstance(params, dict):
            continue
        calls.append(
            {
                "id": f"text_line_{len(calls)}",
                "name": str(name),
                "arguments": params,
                "raw": {
                    "function": {
                        "name": str(name),
                        "arguments": params,
                    }
                },
                "source": "text_fallback",
            }
        )

    pattern = re.compile(
        r'\{\s*"name"\s*:\s*"(?P<name>[^"]+)"\s*,\s*"parameters"\s*:\s*(?P<params>\{.*?\})\s*\}',
        flags=re.DOTALL,
    )

    for match in pattern.finditer(text):
        name = match.group("name")
        params_raw = match.group("params")
        try:
            params = json.loads(params_raw)
        except json.JSONDecodeError:
            continue
        if not isinstance(params, dict):
            continue

        duplicate = any(c["name"] == str(name) and c["arguments"] == params for c in calls)
        if duplicate:
            continue

        calls.append(
            {
                "id": f"text_regex_{len(calls)}",
                "name": str(name),
                "arguments": params,
                "raw": {
                    "function": {
                        "name": str(name),
                        "arguments": params,
                    }
                },
                "source": "text_fallback",
            }
        )

    return calls


def execute_tool(name: str, arguments: dict[str, Any], trace_lines: list[str]) -> dict[str, Any]:
    RUN_STATE["total_tool_calls"] += 1
    call_no = RUN_STATE["total_tool_calls"]

    trace_print("TOOL-CALL", f"#{call_no} {name} args={preview(json.dumps(arguments, ensure_ascii=True))}", trace_lines)

    fn = TOOL_IMPL.get(name)
    if not fn:
        payload = {"ok": False, "error": f"Unknown tool: {name}"}
        trace_print("TOOL-RESULT", f"#{call_no} {name} -> {payload['error']}", trace_lines)
        return payload

    try:
        payload = {"ok": True, "result": fn(**arguments)}
    except Exception as exc:
        payload = {"ok": False, "error": str(exc)}

    trace_print("TOOL-RESULT", f"#{call_no} {name} -> {preview(json.dumps(payload, ensure_ascii=True))}", trace_lines)
    return payload


## Step 4 - Iterative Tool Loop

This is the core orchestration loop:
- request model output with tools
- log content + thinking
- execute tool calls
- append tool outputs back into history
- repeat until completion


In [9]:
# Section: Main loop
# Purpose: Run iterative model/tool turns until the assistant returns a non-tool final answer.

SYSTEM_PROMPT = (
    "You are a DnD dungeon master. Keep narrative vivid but grounded in world state. "
    "Use tools whenever world state lookup/update or dice is needed. "
    "Emit detailed reasoning in your thinking stream and use structured tool calls."
)


def run_tool_loop(user_prompt: str, max_iterations: int = MAX_ITERATIONS) -> dict[str, Any]:
    reset_run_state()
    trace_lines: list[str] = []
    response_snapshots: list[dict[str, Any]] = []
    messages: list[dict[str, Any]] = [{"role": "user", "content": user_prompt}]

    trace_print("RUN", f"model={MODEL} max_iterations={max_iterations}", trace_lines)

    for iteration in range(1, max_iterations + 1):
        trace_print("ITER", f"{iteration}/{max_iterations}", trace_lines)

        try:
            response = adapter.request_with_tools(
                stage="notebook_01_verbose",
                system_prompt=SYSTEM_PROMPT,
                messages=messages,
                tools=TOOLS,
            )
        except LLMError as exc:
            trace_print("MODEL-ERROR", str(exc), trace_lines)
            return {
                "status": "error",
                "error": str(exc),
                "messages": messages,
                "trace_lines": trace_lines,
                "response_snapshots": response_snapshots,
            }

        response_dict = response_to_dict(response)
        response_snapshots.append(response_dict)

        if SHOW_RESPONSE_STATS:
            stats = {
                "model": response_dict.get("model"),
                "done_reason": response_dict.get("done_reason"),
                "prompt_eval_count": response_dict.get("prompt_eval_count"),
                "eval_count": response_dict.get("eval_count"),
                "total_duration": response_dict.get("total_duration"),
            }
            trace_print("MODEL-STATS", json.dumps(stats, ensure_ascii=True), trace_lines)

        if SHOW_RAW_RESPONSE:
            trace_print("RAW-RESPONSE", json.dumps(response_dict, indent=2, ensure_ascii=True), trace_lines)

        assistant_text = adapter._extract_content(response).strip()
        thinking_text = extract_thinking_text(response).strip()

        trace_print("ASSISTANT-CONTENT", preview(assistant_text or "<empty>"), trace_lines)
        if SHOW_THINKING_TRACE:
            trace_print("ASSISTANT-THINKING", preview(thinking_text or "<none>"), trace_lines)

        tool_calls = normalize_tool_calls(response)
        if not tool_calls:
            fallback_source = "".join(x for x in [assistant_text, thinking_text] if x)
            fallback_calls = extract_fallback_tool_calls_from_text(fallback_source)
            if fallback_calls:
                tool_calls = fallback_calls
                trace_print("FALLBACK", f"Recovered {len(tool_calls)} text-encoded tool calls.", trace_lines)

        if not tool_calls:
            messages.append({"role": "assistant", "content": assistant_text})
            return {
                "status": "completed",
                "final_answer": assistant_text,
                "rounds": iteration,
                "messages": messages,
                "trace_lines": trace_lines,
                "response_snapshots": response_snapshots,
                "tool_calls": RUN_STATE["total_tool_calls"],
            }

        messages.append(
            {
                "role": "assistant",
                "content": assistant_text,
                "tool_calls": [c["raw"] for c in tool_calls],
            }
        )

        for call in tool_calls:
            trace_print("TOOL-SOURCE", f"{call['name']} via {call.get('source', 'unknown')}", trace_lines)
            payload = execute_tool(call["name"], call["arguments"], trace_lines)
            messages.append(
                {
                    "role": "tool",
                    "tool_name": call["name"],
                    "content": json.dumps(payload, separators=(",", ":"), ensure_ascii=True),
                }
            )

        trace_print("PROGRESS", f"tool_calls={RUN_STATE['total_tool_calls']}", trace_lines)

    return {
        "status": "max_iterations",
        "final_answer": "Stopped due to max iterations.",
        "messages": messages,
        "trace_lines": trace_lines,
        "response_snapshots": response_snapshots,
        "tool_calls": RUN_STATE["total_tool_calls"],
    }


## Step 5 - Demo Run and Inspection

Run the loop once, print final status, then optionally print the full trace and raw response snapshots.

Tip: start with full trace enabled while learning, then disable for faster runs.


In [10]:
# Section: Demo execution
# Purpose: Execute the loop and print final summary plus optional deep trace dumps.

result = run_tool_loop(
    "The party arrives at Ravenhill and asks if they can track the missing miller through the fog."
)

print("status:", result.get("status"))
print("rounds:", result.get("rounds"))
print("tool_calls:", result.get("tool_calls"))
print()
print("FINAL ANSWER")
print(result.get("final_answer", ""))

trace_lines = result.get("trace_lines", [])
print("\nTRACE LINE COUNT:", len(trace_lines))
if PRINT_FULL_TRACE_AFTER_RUN:
    for line in trace_lines:
        print(line)

snapshots = result.get("response_snapshots", [])
print("\nRESPONSE SNAPSHOT COUNT:", len(snapshots))
if PRINT_FULL_RESPONSE_SNAPSHOTS_AFTER_RUN:
    for i, snap in enumerate(snapshots, start=1):
        print(f"\n--- SNAPSHOT {i} ---")
        print(json.dumps(snap, indent=2, ensure_ascii=True))


[11:14:56] [RUN] model=gpt-oss:20b max_iterations=60
[11:14:56] [ITER] 1/60
[11:15:07] [MODEL-STATS] {"model": "gpt-oss:20b", "done_reason": "stop", "prompt_eval_count": 286, "eval_count": 331, "total_duration": 10304041750}
[11:15:07] [RAW-RESPONSE] {
  "model": "gpt-oss:20b",
  "created_at": "2026-02-20T16:15:07.294224Z",
  "done": true,
  "done_reason": "stop",
  "total_duration": 10304041750,
  "load_duration": 3445962250,
  "prompt_eval_count": 286,
  "prompt_eval_duration": 632490625,
  "eval_count": 331,
  "eval_duration": 6117971300,
  "message": {
    "role": "assistant",
    "content": "",
    "thinking": "We need to respond as DM. The party arrives at Ravenhill. We need to check current scene state? There's no mention of a current world state. But we might want to roll or check for fog? The user asks: \"track the missing miller through the fog.\" We can respond that there is thick fog, possibly making tracking harder. Maybe we need to call roll_dice for perception check? But